In [11]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [12]:
pd.__version__

'2.3.3'

In [13]:
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
# Question 1: Number of columns
print(f"Number of columns: {len(df.columns)}")

Number of columns: 19


In [14]:
!pip install pyarrow


Defaulting to user installation because normal site-packages is not writeable


In [15]:
df.head()



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [16]:
import sklearn

In [17]:
sklearn.__version__

'1.8.0'

In [22]:
# Q2 and Q3
df_train = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df_val = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')
original_train_len = len(df_train)

for df in (df_train, df_val):
    df['duration'] = (pd.to_datetime(df.tpep_dropoff_datetime) - pd.to_datetime(df.tpep_pickup_datetime)).dt.total_seconds() / 60

print(f'Q2 std: {df_train.duration.std():.2f}')

df_train = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)].copy()
df_val = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)].copy()
print(f'Q3 fraction left: {len(df_train) / original_train_len:.4f}')

Q2 std: 42.59
Q3 fraction left: 0.9812


In [23]:
# Q4
categorical = ['PULocationID', 'DOLocationID']
train_q4 = df_train[categorical].copy()
val_q4 = df_val[categorical].copy()

for df in (train_q4, val_q4):
    df[categorical] = df[categorical].astype(str)

dv = DictVectorizer()
X_train = dv.fit_transform(train_q4.to_dict(orient='records'))
X_val = dv.transform(val_q4.to_dict(orient='records'))

print(f'Q4 dimensionality: {X_train.shape[1]}')

Q4 dimensionality: 515


In [24]:
# Q5
y_train = df_train['duration'].values
lr = LinearRegression()
lr.fit(X_train, y_train)
train_pred = lr.predict(X_train)

print(f'Q5 RMSE train: {root_mean_squared_error(y_train, train_pred):.2f}')

Q5 RMSE train: 7.65


In [25]:
# Q6
y_val = df_val['duration'].values
val_pred = lr.predict(X_val)

print(f'Q6 RMSE validation: {root_mean_squared_error(y_val, val_pred):.2f}')

Q6 RMSE validation: 7.81
